# NDVI con Sentinel-2 y Google Earth Engine + ArcGIS Pro

**Material del Webinar introductorio a Google Earth Engine (GEE)**
**Autoría:** Grettel Vargas


> La **mediana** se calcula píxel a píxel usando todas las imágenes seleccionadas. Es útil en una demostración porque genera una imagen representativa del período y reduce la influencia de valores extremos, como nubes residuales.


## 1. Librerías
Carga las librerías necesarias para trabajar con **Google Earth Engine**, visualizar mapas y exportar resultados.


In [ ]:
# Si geemap no está instalado, quitar el # de la siguiente línea y ejecutar esta celda.
# %pip install geemap


In [14]:
import os
import ee
import geemap
import arcpy

print("✓ Librerías cargadas")


✓ Librerías cargadas


## 2. Parámetros
Define los valores que se pueden modificar fácilmente:

- `PAIS` y `DEPTO`: área de estudio.
- `FECHA_INICIO` y `FECHA_FIN`: período de análisis.
- `MAX_NUBES`: porcentaje máximo de nubosidad permitido.
- `ESCALA_M`: resolución espacial de salida en metros.
- `CRS_SALIDA`: sistema de coordenadas del GeoTIFF.


In [15]:
PROYECTO_GEE = 'ee-gretapuravida'

PAIS = 'Guatemala'
DEPTO = 'Zacapa'

FECHA_INICIO = '2026-01-01'
FECHA_FIN = '2026-04-01' # fecha final exclusiva
MAX_NUBES = 20

ESCALA_M = 30
CRS_SALIDA = 'EPSG:32616'


## 3. Conectar con Earth Engine
Inicia la conexión con **Google Earth Engine** utilizando el proyecto definido en `PROYECTO_GEE`.


In [16]:
try:
 ee.Initialize(project=PROYECTO_GEE)
except Exception:
 ee.Authenticate()
 ee.Initialize(project=PROYECTO_GEE)

print("✓ Google Earth Engine conectado")


✓ Google Earth Engine conectado


## 4. Área de estudio
Selecciona el departamento indicado en los parámetros utilizando los límites administrativos de **FAO GAUL**.


In [17]:
limites = ee.FeatureCollection('FAO/GAUL/2015/level1')

depto = limites.filter(
 ee.Filter.And(
 ee.Filter.eq('ADM0_NAME', PAIS),
 ee.Filter.eq('ADM1_NAME', DEPTO)
 )
)

geom = depto.geometry()
lon, lat = geom.centroid().coordinates().getInfo()

print(f"✓ Área de estudio: {DEPTO}, {PAIS}")


✓ Área de estudio: Zacapa, Guatemala


## 5. Sentinel-2 y compuesto por mediana
Selecciona imágenes **Sentinel-2** según el área, las fechas y la nubosidad.

`median()` calcula la mediana de los valores disponibles para cada píxel y genera una sola imagen representativa del período.


In [18]:
coleccion = (
 ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
 .filterBounds(geom)
 .filterDate(FECHA_INICIO, FECHA_FIN)
 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_NUBES))
)

print("Número de imágenes:", coleccion.size().getInfo())

# Compuesto representativo del período
s2_mediana = coleccion.median().clip(geom)

print("✓ Compuesto por mediana creado")


Número de imágenes: 6
✓ Compuesto por mediana creado


## 6. Calcular NDVI
Calcula el **Índice de Vegetación de Diferencia Normalizada (NDVI)** utilizando:

- `B8`: infrarrojo cercano.
- `B4`: rojo.

El NDVI permite representar el vigor o verdor de la vegetación.


In [19]:
ndvi = s2_mediana.normalizedDifference(['B8', 'B4']).rename('NDVI')

print("✓ NDVI calculado")


✓ NDVI calculado


## 7. Visualizar en el notebook
Muestra el compuesto Sentinel-2 y el NDVI sobre **Esri World Imagery**.

Los parámetros `min`, `max` y `palette` controlan cómo se representan los valores en el mapa.


In [20]:
Map = geemap.Map(
 center=[lat, lon],
 zoom=9,
 basemap='Esri.WorldImagery'
)

Map.addLayer(
 s2_mediana,
 {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
 'Sentinel-2 - mediana'
)

vis_ndvi = {
 'min': 0,
 'max': 0.8,
 'palette': ['red', 'yellow', 'green']
}

Map.addLayer(ndvi, vis_ndvi, 'NDVI')
Map.addLayer(depto.style(color='black', fillColor='00000000'), {}, 'Límite')

Map


Map(center=[15.037018921487363, -89.50839485004906], controls=(WidgetControl(options=['position', 'transparent…

## 8. Exportar y agregar a ArcGIS Pro
Exporta los resultados como **GeoTIFF** y los agrega al mapa activo de ArcGIS Pro.

La resolución y el sistema de coordenadas se toman de `ESCALA_M` y `CRS_SALIDA`.


In [21]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.activeMap

carpeta = os.path.join(aprx.homeFolder, "NDVI_export")
os.makedirs(carpeta, exist_ok=True)

ruta = os.path.join(carpeta, f"NDVI_{DEPTO}.tif")

geemap.ee_export_image(
 ndvi.toFloat(),
 filename=ruta,
 scale=ESCALA_M,
 crs=CRS_SALIDA,
 region=geom,
 file_per_band=False
)

if mapa is not None:
 mapa.addDataFromPath(ruta)

print("✓ GeoTIFF exportado y agregado al mapa:")
print(ruta)


Generating URL ...
Please wait ...
Data downloaded to C:\Users\GRETTEL\OneDrive\Desktop\Respaldos\FAO\Datos geoespaciales\Honduras\Restauracion_Forestal\Set_data_ICF\NDVI_export\NDVI_Zacapa.tif
✓ GeoTIFF exportado y agregado al mapa:
C:\Users\GRETTEL\OneDrive\Desktop\Respaldos\FAO\Datos geoespaciales\Honduras\Restauracion_Forestal\Set_data_ICF\NDVI_export\NDVI_Zacapa.tif
